# 01 · 预训练：让 MiniMind 学会预测下一个 token

**本节目标**

1. 看懂 `{"text": "..."}` 数据怎样变成模型输入与监督信号。
2. 对照 MiniMind 原生 `train_pretrain.py`，认识模型、交叉熵损失、优化器与检查点。
3. 在一张 24 GB GPU 上从随机初始化训练 512 条样本，得到独立的练习权重。

**预计资源**：单卡 CUDA GPU；首次下载 `pretrain_t2t_mini.jsonl` 约 1.2 GB；训练仅使用其中最多 512 条。短实验以完成一次训练链路为目标，不追求可对话的模型质量。

代码来源：[MiniMind](https://github.com/jingyaogong/minimind)；数据来源：[MiniMind 数据集（ModelScope）](https://modelscope.cn/datasets/gongjy/minimind_dataset)。

## 1. 确认 Notebook 工作目录与 GPU

ModelScope Gallery 将课程复制到 Notebook 工作空间后，工作目录可能与 Notebook 所在目录不同。下面会从当前目录及其父目录寻找本课程。找不到时，请把 `COURSE_ROOT` 改成实际复制路径。

In [ ]:
from pathlib import Path
import importlib.util
import shutil
import subprocess
import sys
import os
import threading
import torch

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
candidates += [p.parent.parent for p in Path.cwd().glob("*/notebooks/01_pretrain.ipynb")]
candidates += [p.parent.parent for p in Path.cwd().glob("*/*/notebooks/01_pretrain.ipynb")]
COURSE_ROOT = next((p for p in candidates if (p / "notebooks" / "01_pretrain.ipynb").is_file()), None)
if COURSE_ROOT is None:
    raise FileNotFoundError("请将 COURSE_ROOT 手动设为课程仓库根目录，再运行此单元")
WORK = COURSE_ROOT / "work"
WORK.mkdir(exist_ok=True)
print("课程目录:", COURSE_ROOT)
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("本节短训练需要 CUDA GPU；请在 ModelScope Notebook 中启动 GPU 实例")
print("GPU:", torch.cuda.get_device_name(0))
print("显存 GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## 2. 准备 MiniMind 源码和依赖

本节直接调用上游的原生 PyTorch 训练脚本。源码从 GitHub 克隆到 `work/minimind`。这里不会下载已经训练好的 MiniMind 权重：预训练从随机参数开始。

ModelScope Notebook 的预装包可能随镜像变化。先检查依赖，缺失时在新单元运行 `%pip install modelscope datasets transformers sentencepiece`，随后重启内核并从头执行。不要为了本节重装 PyTorch。

In [ ]:
required = ("modelscope", "datasets", "transformers", "sentencepiece")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"缺少 {missing}；请按上方说明安装后重启内核")
SOURCE = WORK / "minimind"
if not (SOURCE / "trainer" / "train_pretrain.py").is_file():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/jingyaogong/minimind.git", str(SOURCE)],
        check=True,
    )
print("源码:", SOURCE)


## 3. 从 ModelScope 获取预训练数据

数据集每行是一个 JSON 对象，包含 `text` 字段。本节只下载需要的 `pretrain_t2t_mini.jsonl`。若你在课程目录的同级已经下载过它，可直接复用，避免重复下载。

In [ ]:
from modelscope.hub.file_download import dataset_file_download

dataset_name = "pretrain_t2t_mini.jsonl"
nearby = COURSE_ROOT.parent / "minimind_dataset" / dataset_name
cached = WORK / "datasets" / dataset_name
if nearby.is_file():
    DATA_FILE = nearby
elif cached.is_file():
    DATA_FILE = cached
else:
    cached.parent.mkdir(parents=True, exist_ok=True)
    DATA_FILE = Path(dataset_file_download(
        dataset_id="gongjy/minimind_dataset",
        file_path=dataset_name,
        local_dir=str(cached.parent),
    ))
if not DATA_FILE.is_file():
    raise FileNotFoundError(DATA_FILE)
print("数据文件:", DATA_FILE, "大小 GiB:", round(DATA_FILE.stat().st_size / 2**30, 2))


## 4. 理解 next-token 训练目标

假设 token 序列是 `[A, B, C, D]`：模型看到 `[A, B, C]`，分别预测 `[B, C, D]`。训练时不会直接拿整段文字与输出做字符串比较，而是计算每个目标 token 的交叉熵，再对有效位置求平均。

MiniMind 的 `PretrainDataset` 负责把文本编码、截断、padding，训练循环把 `input_ids` 与 `labels` 送入模型；`model(..., labels=labels)` 返回损失。先看一条原始样本和分词结果。

In [ ]:
import json
from transformers import AutoTokenizer

with DATA_FILE.open("r", encoding="utf-8") as f:
    first = json.loads(next(f))
assert isinstance(first.get("text"), str) and first["text"].strip(), "数据格式应包含非空 text 字段"
tokenizer = AutoTokenizer.from_pretrained(str(SOURCE / "model"))
token_ids = tokenizer(first["text"], add_special_tokens=False)["input_ids"]
print("文本片段:", first["text"][:160])
print("token 数:", len(token_ids))
print("前 20 个 token ID:", token_ids[:20])
print("预测示意: 输入", token_ids[:8], "→ 目标", token_ids[1:9])


## 5. 生成短实验数据

为了让课程实验迅速完成，这里取文件开头最多 512 条有效样本。它们不是随机样本，因此不能用于评估模型质量。输出仍使用上游脚本要求的 JSONL 格式。

In [ ]:
import itertools

SMALL_DATA = WORK / "datasets" / "pretrain_course_512.jsonl"
SMALL_DATA.parent.mkdir(parents=True, exist_ok=True)
count = 0
with DATA_FILE.open("r", encoding="utf-8") as src, SMALL_DATA.open("w", encoding="utf-8") as dst:
    for line in itertools.islice(src, 512):
        row = json.loads(line)
        if isinstance(row.get("text"), str) and row["text"].strip():
            dst.write(json.dumps({"text": row["text"]}, ensure_ascii=False) + "\n")
            count += 1
assert count > 0
print("练习样本:", count, "文件:", SMALL_DATA)


## 6. 单卡运行上游预训练脚本

默认 64M Dense 结构为 `hidden_size=768`、`num_hidden_layers=8`。本实验用 `batch_size=8`、梯度累积 4 步，有效 batch 为 32，`max_seq_len=340`，训练 1 个 epoch。权重保存到课程独立目录，名称为 `course_pretrain_768.pth`，不会覆盖你已有的 `pretrain_768.pth`。

这是从头训练：`--from_weight none`。日志中的 loss 反映 next-token 任务，不是聊天质量分数。

训练单元还会在 `nvidia-smi` 可用时每 0.5 秒记录一次当前 GPU 的整卡显存使用量；多卡服务器可在启动 Notebook 前设置 `CUDA_VISIBLE_DEVICES=6`。整卡读数包括同卡其他进程。

In [ ]:
OUT = WORK / "runs" / "pretrain"
OUT.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, "train_pretrain.py",
    "--data_path", str(SMALL_DATA),
    "--save_dir", str(OUT),
    "--save_weight", "course_pretrain",
    "--from_weight", "none",
    "--epochs", "1",
    "--batch_size", "8",
    "--accumulation_steps", "4",
    "--max_seq_len", "340",
    "--num_workers", "2",
    "--log_interval", "10",
    "--save_interval", "10000",
]
print("运行:", " ".join(cmd))
gpu_visible = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]
gpu_index = gpu_visible if gpu_visible.isdigit() else "0"
readings_mib = []
stop_monitor = threading.Event()
def watch_gpu():
    if shutil.which("nvidia-smi") is None:
        return
    while not stop_monitor.is_set():
        result = subprocess.run(
            ["nvidia-smi", f"--id={gpu_index}", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            try:
                readings_mib.append(int(result.stdout.strip().splitlines()[0]))
            except (ValueError, IndexError):
                pass
        stop_monitor.wait(0.5)
monitor = threading.Thread(target=watch_gpu, daemon=True)
monitor.start()
try:
    subprocess.run(cmd, cwd=SOURCE / "trainer", check=True)
finally:
    stop_monitor.set()
    monitor.join(timeout=3)
if readings_mib:
    print("整卡显存峰值 MiB:", max(readings_mib), "约 GiB:", round(max(readings_mib) / 1024, 2))
else:
    print("无法从 nvidia-smi 读取显存；可在终端运行 nvidia-smi 观察")


## 7. 检查训练产物

上游脚本在 epoch 最后一步保存权重。这里检查文件、参数数量和数值有限性；这些检查能确认训练链路完整，但不能证明模型已经具备对话能力。

In [ ]:
WEIGHT = OUT / "course_pretrain_768.pth"
assert WEIGHT.is_file(), f"未找到训练权重: {WEIGHT}"
weights = torch.load(WEIGHT, map_location="cpu", weights_only=True)
parameter_count = sum(t.numel() for t in weights.values())
all_finite = all(torch.isfinite(t).all().item() for t in weights.values())
print("权重:", WEIGHT)
print("大小 MiB:", round(WEIGHT.stat().st_size / 2**20, 1))
print("参数量 M:", round(parameter_count / 1e6, 2))
print("参数均为有限数:", all_finite)
assert all_finite


## 回顾与练习

- **预训练学到了什么？** 在大量文本上优化“下一个 token”的概率分布；它尚未接受专门的指令跟随训练。
- **为什么这里不能用回答质量评判？** 512 条样本只够演示训练管线，无法得到通用聊天模型。
- **怎么扩展为完整实验？** 改用完整 `pretrain_t2t_mini.jsonl`，按上游推荐参数训练，并记录硬件、数据版本、epoch、loss 与权重路径。MiniMind 官方给出了单张 RTX 3090 运行 mini 数据的经验时长；实际耗时以你的环境为准。
- **思考题**：若 `batch_size` 减半而希望有效 batch 不变，`accumulation_steps` 应如何调整？为何训练序列变长会增加显存占用？

下一节将从这类预训练权重出发，解释 SFT 如何把语言建模能力变为指令响应能力。完整流程与参数请对照 [MiniMind 预训练脚本](https://github.com/jingyaogong/minimind/blob/master/trainer/train_pretrain.py)。